# Layer 3 Part A: Stage 1 — Deontic Decomposition 

**Scope:** Stage 1 only. No Ontology extraction, no validation.



In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'together', 'nbformat'])
print('Dependencies ready.')

In [ ]:
import json
import os
import random
import re
import time
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
from together import Together
print('Imports OK')

In [ ]:
os.environ['TOGETHER_API_KEY'] = '' 
if 'PASTE_YOUR' in os.environ['TOGETHER_API_KEY']:
    raise ValueError(
        'Replace PASTE_YOUR_TOGETHER_KEY_HERE with your actual Together API key.'
    )
print('Together API key set.')

In [ ]:
# -- Model configuration ------------------------------------------------------
GENERATION_MODEL = 'deepseek-ai/DeepSeek-V4-Pro'

# -- Paths --------------------------------------------------------------------
BASE_DIR = Path('/Users/umair/CCO-GRO')

RETRIEVAL_DIR = BASE_DIR / '2 - Retrieval Layer' / 'output' / 'retrieval_contexts'
RETRIEVAL_FILES = {
    'UK': RETRIEVAL_DIR / 'retrieval_uk.json',
    'Canada': RETRIEVAL_DIR / 'retrieval_canada.json',
    'Australia': RETRIEVAL_DIR / 'retrieval_australia.json',
    'USA': RETRIEVAL_DIR / 'retrieval_usa.json',
}

# Full corpus run output (separate from pilot)
OUTPUT_DIR = BASE_DIR / '3 - Extraction and  Validation Layer' / 'output' / 'stage1_decomposition'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -- Runtime configuration ----------------------------------------------------
RANDOM_SEED = 25
MAX_SCHEMA_HITS_IN_PROMPT = 5
MAX_ODP_HITS_IN_PROMPT = 3
GEN_TEMPERATURE = 0.1

# Full corpus run flags
VERBOSE = False              # Set True for per-chunk debug output
PROGRESS_EVERY = 25          # Print progress every N chunks
CHECKPOINT_INTERVAL = 50     # Save checkpoint every N chunks
MAX_API_RETRIES = 3          # Retry transient API failures

print('Configuration OK')
print(f'  Generation model        : {GENERATION_MODEL}')
print(f'  Retrieval dir           : {RETRIEVAL_DIR}')
print(f'  Output dir              : {OUTPUT_DIR}')
print(f'  Mode                    : FULL CORPUS (no sampling)')
print(f'  Checkpoint interval     : {CHECKPOINT_INTERVAL}')
print(f'  Max API retries         : {MAX_API_RETRIES}')

In [ ]:
_client = None

def get_client():
    global _client
    if _client is None:
        _client = Together()  # reads TOGETHER_API_KEY from env
    return _client

def call_stage1_llm(system: str, user: str, max_tokens: int = 1400, temperature: float = GEN_TEMPERATURE) -> str:
    """Stage 1: Deontic Decomposition via DeepSeek-V4-Pro (thinking disabled)."""
    response = get_client().chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': user},
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        reasoning={"enabled": False},   # ← Critical: disable thinking for V4-Pro
    )
    return (response.choices[0].message.content or '').strip()

def call_stage1_llm_with_retry(system: str, user: str, max_retries: int = MAX_API_RETRIES,
                                max_tokens: int = 1400, temperature: float = GEN_TEMPERATURE) -> str:
    """Wrapper with exponential backoff retry for transient API failures."""
    last_error = None
    for attempt in range(max_retries):
        try:
            return call_stage1_llm(system, user, max_tokens, temperature)
        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                wait = 2 ** attempt  # 1s, 2s, 4s
                print(f'    API error (attempt {attempt+1}/{max_retries}), retry in {wait}s: {e}')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f'All {max_retries} retries failed: {last_error}')

print('LLM client wrapper defined (with retry).')

In [ ]:
# -- Reference constants for Stage 1 output validation ------------------------
# All values mirror CCO.ttl and odp_library_layer3_final.json.

# ODP labels — all 12 patterns from odp_library_layer3_final.json
ODP_LABELS = {
    'Obligation Pattern', 'Permission Pattern', 'Prohibition Pattern',
    'Regulatory Authority Pattern', 'Conditional Norm Pattern', 'Exception Pattern',
    'Funding Allocation Pattern', 'Role-Bearing Subject Pattern',
    'Norm Temporal Applicability Pattern', 'Role-Holding Temporal Pattern',
    'Regulatory Supersession Pattern', 'Action Pattern',
}

# Subject CCO types — Agent hierarchy + Role hierarchy + fallback
SUBJECT_CCO_TYPES = {
    'Agent', 'Person', 'Organisation', 'RegulatoryAuthorityAgent',
    'Role', 'RegulatoryAuthorityRole',
    'Unknown'
}

# Deontic types — Norm subclasses + Norm itself + Exception
DEONTIC_TYPES = {
    'Obligation', 'Permission', 'Prohibition',  # Norm subclasses
    'Norm',                                       # Generic parent (fallback)
    'Exception',                                  # Norm-modifying (not Norm subclass)
    None,                                         # For non_extractable cases
}

print(f'ODP labels available    : {len(ODP_LABELS)}')
print(f'Subject CCO types       : {len(SUBJECT_CCO_TYPES)}')
print(f'Deontic types accepted  : {len([d for d in DEONTIC_TYPES if d is not None])} + None')

# -- Pronoun handling constants (v2.5) ----------------------------------------
PRONOUN_SET = {
    'you', 'your', 'yours', 'yourself',
    'he', 'him', 'his', 'himself',
    'she', 'her', 'hers', 'herself',
    'they', 'them', 'their', 'theirs', 'themselves',
    'it', 'its', 'itself',
    'we', 'us', 'our', 'ours', 'ourselves',
    'i', 'me', 'my', 'mine', 'myself',
    'one', 'oneself',
}

# Document-level addressee dictionary
DOCUMENT_ADDRESSEE = {
    'Australia': 'approved course provider',
    'USA':       'institution',
    # 'UK':      None,   # no pronoun cases in pilot
    # 'Canada':  None,   # no pronoun cases in pilot
}

print(f'Pronoun tokens tracked  : {len(PRONOUN_SET)}')
print(f'Document addressee map  : {len(DOCUMENT_ADDRESSEE)} jurisdictions')

In [ ]:
# -- JSON parsing helpers -----------------------------------------------------
def _extract_first_json_object(text: str) -> str:
    start = text.find('{')
    if start == -1:
        raise ValueError('No JSON object start found in model response')
    depth = 0
    in_string = False
    escape = False
    for idx in range(start, len(text)):
        ch = text[idx]
        if in_string:
            if escape:
                escape = False
            elif ch == '\\':
                escape = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == '{':
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    return text[start:idx + 1]
    raise ValueError('No complete JSON object found in model response')

def clean_model_json_text(raw: str) -> str:
    cleaned = (raw or '').strip()
    if not cleaned:
        raise ValueError('Empty model response')
    cleaned = re.sub(r'<think>.*?</think>', '', cleaned, flags=re.S | re.I).strip()
    if cleaned.startswith('```'):
        lines = [line for line in cleaned.splitlines() if not line.strip().startswith('```')]
        cleaned = '\n'.join(lines).strip()
    if not cleaned:
        raise ValueError('Model response empty after cleanup')
    if not cleaned.lstrip().startswith('{'):
        cleaned = _extract_first_json_object(cleaned)
    return cleaned

def parse_json_response(raw: str):
    cleaned = clean_model_json_text(raw)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        extracted = _extract_first_json_object(cleaned)
        return json.loads(extracted)

def short_text(text: str, limit: int = 220) -> str:
    text = re.sub(r'\s+', ' ', text).strip()
    return text if len(text) <= limit else text[:limit].rstrip() + '...'

# -- Prompt formatting helpers ------------------------------------------------
def format_schema_hits(schema_hits: list, top_k: int = MAX_SCHEMA_HITS_IN_PROMPT) -> str:
    hits = schema_hits[:top_k]
    if not hits:
        return 'No schema hits available.'
    lines = []
    for hit in hits:
        lines.append(
            f"- {hit['item_id']} | {hit['item_type']} | {hit['label']} | score={hit['score']:.4f}\n"
            f"  {short_text(hit.get('text', ''))}"
        )
    return '\n'.join(lines)

def format_odp_hits(odp_hits: list, top_k: int = MAX_ODP_HITS_IN_PROMPT) -> str:
    hits = odp_hits[:top_k]
    if not hits:
        return 'No ODP hits available.'
    lines = []
    for hit in hits:
        lines.append(
            f"- {hit['item_id']} | {hit['label']} | score={hit['score']:.4f}\n"
            f"  {short_text(hit.get('text', ''))}"
        )
    return '\n'.join(lines)

# -- Data loading -------------------------------------------------------------
def load_final_retrieval_contexts():
    """Load all retrieval contexts marked selected_for_llm."""
    all_contexts = []
    for jurisdiction, path in RETRIEVAL_FILES.items():
        with open(path, encoding='utf-8') as f:
            payload = json.load(f)
        for ctx in payload['retrieval_contexts']:
            if ctx.get('selected_for_llm'):
                all_contexts.append(ctx)
    return all_contexts

def save_json(path: Path, payload: dict):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

print('Helpers + data loaders defined.')

In [ ]:
# -- Stage 1 Prompt: Regulatory Provision Decomposition for Education Funding ----
# v3.1 — Quality patches: anti-paraphrasing strengthening, fragment filter, modal verb gate

STAGE1_SYSTEM = """You are a regulatory provision analyst working on an education funding regulatory ontology.
Your job is to extract structured ontology-mappable content from education funding regulation chunks.
The source documents are funding rules, eligibility criteria, allocation policies, and procedural rules from UK, Canada, Australia, and USA education funding regulators.
Capture ANY provision that establishes a regulatory fact about funding, eligibility, allocation, calculation, definition, or duty — not only sentences with explicit modal verbs.
Respond with valid JSON only. Use JSON null (not the string \"null\") for empty fields."""

STAGE1_USER_TEMPLATE = """Analyze this retrieved unit from an education funding regulation.
PROVISION ID: {unit_id}
JURISDICTION: {jurisdiction}
UNIT TYPE: {unit_type}
SECTION PATH: {section_path}
PROVISION TEXT:
{text}
RETRIEVED CCO SCHEMA HITS:
{schema_hits}
RETRIEVED ODP HITS:
{odp_hits}
Return this exact JSON shape:
{{
  "extraction_decision": "extractable|non_extractable",
  "non_extractable_reason": "short reason or null",
  "provision_type": "Obligation|Permission|Prohibition|FundingAllocation|EligibilityRule|CalculationRule|Definition|Exception|null",
  "deontic_type": "Obligation|Permission|Prohibition|Norm|Exception|null",
  "norm_statement": "short paraphrase or null",
  "subject": {{
    "text": "actor, bearer, or named entity in the provision (use literal text from source; for definitions, use the defined term)",
    "suggested_cco_type": "Organisation|Person|Agent|RegulatoryAuthorityAgent|Role|RegulatoryAuthorityRole|Resource|Unknown"
  }},
  "action_or_state": "what must/may/must not happen, what is allocated, what is defined, or null",
  "object_or_resource": "resource, thing, or activity involved, else null",
  "monetary_amount": "any explicit funding amount, rate, or cap in the provision, else null",
  "condition_text": "if/when/where/provided that clause, else null",
  "exception_text": "unless/except/notwithstanding clause, else null",
  "temporal_text": "date, deadline, timing cue, else null",
  "authority_text": "issuing or enforcing authority if present, else null",
  "source_regulation_text": "regulation or instrument name if explicit, else null",
  "likely_primary_odp_label": "plain ODP label (e.g., 'Funding Allocation Pattern'), never include 'ODP-x |' prefix",
  "likely_secondary_odp_labels": ["plain ODP labels only, no prefixes"],
  "extraction_rationale": "1-2 sentence explanation grounded in the provision"
}}

SCOPE — What this ontology captures:
This is an Education Funding Regulatory Ontology. We extract any provision that establishes a regulatory fact about education funding, including:
  (1) Deontic norms (Obligations, Permissions, Prohibitions on named actors)
  (2) Funding allocations (amounts, rates, caps, instalments, who receives what)
  (3) Eligibility rules (who qualifies, what criteria apply)
  (4) Calculation rules (how amounts/durations/contributions are computed)
  (5) Allowable / ineligible costs (what counts as fundable expense)
  (6) Definitions of regulatory terms (what counts as 'exceptional expense', 'discretionary income', etc.)
  (7) Procedural rules (when funds are disbursed, how applications are processed)
  (8) Exceptions and conditions modifying any of the above

Rules:

1. EXTRACTION SCOPE — Be inclusive of regulatory content, exclusive of non-content:
   Mark `extractable` if the chunk contains ANY of:
   - A modal verb (must, shall, may, may not) with any sensible subject — actor OR resource
   - An explicit funding amount, rate, cap, percentage, or formula
   - An eligibility criterion (who qualifies, age, status, condition)
   - A definition of a regulatory term
   - A calculation procedure (passive voice acceptable here)
   - A list of allowable or ineligible items/costs/activities
   - A statement establishing a funding allocation between entities
   - A timing or procedural rule about funding disbursement

   Mark `non_extractable` only if the chunk is:
   - Pure editorial/navigational text ("See Chapter 8", "Continued on next page")
   - Document metadata ("Last updated: March 2025")
   - Table column headers without data ("| Type | Amount |")
   - Sentence fragments truncated mid-thought with no recoverable meaning
   - Cross-reference pointers with no substantive content
   - Pure transitional sentences ("This chapter outlines...", "In summary,...")
   - Policy announcements describing past events ("In February 2025, the Government announced...")
   - Bullet/list fragments missing their parent context (e.g., "• a declaration by..." with no surrounding subject)
   - Table cell fragments with broken syntax (e.g., "| | 60 penalty units | | |")
   - Phrases describing past states (e.g., "The provider has confirmed that...")
   - Pure background commentary without normative force ("Another factor is...")

2. PROVISION TYPE — Classify the dominant regulatory pattern in this chunk:
   - "Obligation": Named actor must/shall do X. Modal verb + named regulated party.
     Example: "The provider must retain evidence of attendance for six years."
   - "Permission": Named actor may/can do X, or is entitled to X.
     Example: "The Secretary may recredit a student's HELP balance."
   - "Prohibition": Named actor must not / shall not / may not do X.
     Example: "Providers must not charge above the fee cap."
   - "FundingAllocation": A monetary resource is allocated to a recipient at a stated amount/rate.
     Example: "Learning support funding is fixed at £150 per month."
     Example: "Students with dependents may receive maximum of $740 per week."
   - "EligibilityRule": A condition determines who qualifies for funding/program/role.
     Example: "Students who are incarcerated are not eligible for funding."
     Example: "The program must be at least 12 weeks in duration."
   - "CalculationRule": A formula or procedure computes an amount, duration, or status.
     Example: "Multiply the number of months by 4.3 to determine study weeks."
     Example: "Discretionary income is total income less taxes, CPP/QPP and EI."
   - "Definition": Establishes meaning of a regulatory term used elsewhere in the framework.
     Example: "Exceptional expenses are those not normally associated with attending school."
     Example: "Non-punitive Withdrawal: withdrawal before classes start where..."
   - "Exception": Modifies a previously stated rule under specific conditions.
     Example: "Except where the regulator has granted a waiver."

3. DEONTIC TYPE for downstream CCO mapping:
   Even when `provision_type` is FundingAllocation, EligibilityRule, CalculationRule, or Definition, ALWAYS attempt to set `deontic_type` according to the underlying deontic force:
   - FundingAllocation → typically `Permission` (recipients are entitled to receive) or `Obligation` (allocator must pay)
   - EligibilityRule → typically `Permission` (those who qualify may participate) or `Prohibition` (those who don't are excluded)
   - CalculationRule → typically `Obligation` (the calculation must be applied this way) or `null` if purely procedural
   - Definition → typically `null` (definitions are not directly deontic) — set to null is acceptable here
   - Exception → typically `Exception`

   Use `Norm` only as a last resort if none of Obligation/Permission/Prohibition/Exception fit.

4. SUBJECT — Use the most regulatory-meaningful subject:
   Acceptable subject types for different provision_types:
   - Obligation/Permission/Prohibition: the named actor (provider, student, employer, authority)
   - FundingAllocation: EITHER the recipient ("students with dependents") OR the resource itself ("learning support funding") — both are acceptable; prefer the recipient when named
   - EligibilityRule: the entity being qualified ("the program", "students under 19", "the institution")
   - CalculationRule: the variable being calculated ("discretionary income", "study period in weeks")
   - Definition: the term being defined ("exceptional expenses", "non-punitive withdrawal")
   - Exception: the rule being modified, or the entity exempted

   For `subject.suggested_cco_type`:
   - Use 'Organisation', 'Person', 'Agent', 'RegulatoryAuthorityAgent', 'Role', or 'RegulatoryAuthorityRole' when the subject is a regulated party
   - Use 'Resource' when the subject is a funding resource, allocation, or monetary item (only valid for FundingAllocation provision_type)
   - Use 'Unknown' only when subject is genuinely ambiguous AND is a regulated party (do not use as escape hatch for non-actors)

5. PRONOUN HANDLING:
   If the subject of the norm is a pronoun (you, your, he, she, they, their, we, our, it, etc.), preserve the pronoun EXACTLY as it appears in the provision text. Do NOT invent a resolved subject name. Set `subject.text` to the literal pronoun and set `subject.suggested_cco_type` to `"Role"`. Pronoun resolution is handled deterministically by a downstream post-processor.

6. NO HALLUCINATION — STRICT FAITHFULNESS TO SOURCE TEXT:
   Only extract content actually present in the provision text. Do not invent temporal markers, authorities, conditions, or exceptions that are not stated.

   CRITICAL — PASSIVE VERB PRESERVATION:
   Do NOT paraphrase passive verbs or non-modal verbs into active modal duties. The norm_statement field MUST preserve the verb form of the source text. If you change "X aligns with Y" into "X must align with Y", you are violating this rule.
   
    FORBIDDEN paraphrasing patterns:
   - "X aligns with Y" → "X must align with Y"  (preserve "aligns")
   - "X has confirmed Y" → "X must confirm Y"  (preserve "has confirmed"; this is past tense, not a duty)
   - "X is calculated by Y" → "X must be calculated by Y"  (preserve "is calculated")
   - "X is assessed according to Y" → "X must be assessed according to Y"  (preserve "is assessed")
   - "X are expected to do Y" → deontic_type=Obligation  ("expected to" is advisory, NOT mandatory)
   - "X is included in Y" → "X must be included in Y"  (preserve "is included")

   ADVISORY vs MANDATORY language:
   The following phrases are ADVISORY/RECOMMENDED — do NOT classify them as Obligation:
   - "are expected to", "is expected to", "expected"
   - "should", "is recommended", "is encouraged"
   - "is intended to", "aims to", "is designed to"
   If advisory: use deontic_type=Permission (if it confers ability) or null (if purely descriptive).

   PAST-TENSE STATEMENTS:
   The following past-tense patterns describe completed states, NOT duties — do NOT classify as Obligation:
   - "X has confirmed Y", "X has provided Y", "X has met Y"
   - "X was approved", "X has been issued"
   If the chunk only describes a past state without imposing a future duty: mark non_extractable OR Definition (if defining the state).

7. CCO TYPE WHITELIST — Strict enforcement:
   `subject.suggested_cco_type` MUST be EXACTLY one of:
   - Organisation, Person, Agent, RegulatoryAuthorityAgent, Role, RegulatoryAuthorityRole, Resource, Unknown
   FORBIDDEN: 'Document', 'Process', 'System', 'Definition', 'Calculation', or any other CCO class not listed above.

8. ODP WHITELIST — Strict enforcement:
   `likely_primary_odp_label` MUST be EXACTLY one of these 12 patterns:
   - Obligation Pattern
   - Permission Pattern
   - Prohibition Pattern
   - Exception Pattern
   - Regulatory Authority Pattern
   - Conditional Norm Pattern
   - Funding Allocation Pattern
   - Role-Bearing Subject Pattern
   - Norm Temporal Applicability Pattern
   - Role-Holding Temporal Pattern
   - Regulatory Supersession Pattern
   - Action Pattern

   ODP-to-provision_type mapping guidance:
   - Obligation → Obligation Pattern
   - Permission → Permission Pattern
   - Prohibition → Prohibition Pattern
   - FundingAllocation → Funding Allocation Pattern (primary), with Permission/Obligation Pattern as secondary when actor named
   - EligibilityRule → Conditional Norm Pattern (primary), with Role-Bearing Subject Pattern as secondary
   - CalculationRule → Action Pattern (primary), with Conditional Norm Pattern as secondary if conditional
   - Definition → Role-Bearing Subject Pattern (primary) if defining a role; else Action Pattern
   - Exception → Exception Pattern

   FORBIDDEN ODP values: 'Norm Pattern', 'Generic Pattern', 'Rule Pattern', 'Calculation Pattern', 'Definition Pattern' — these do NOT exist.

9. CONDITIONAL HANDLING:
   If a provision is conditional ("If X, then Y..."), place the conditional clause in `condition_text` and set `provision_type`/`deontic_type` based on the underlying rule that applies when condition is met. Add 'Conditional Norm Pattern' as a secondary ODP label.

10. MONETARY AMOUNT EXTRACTION:
    For ANY provision mentioning a specific funding amount, rate, cap, percentage, or threshold (£150, $740, 50%, 30 days, etc.), capture it in `monetary_amount`. This is critical for the funding ontology. Examples:
    - "fixed at £150 per month" → monetary_amount: "£150 per month"
    - "maximum of $740 per week" → monetary_amount: "maximum $740 per week"
    - "12 weeks in duration within 15 weeks" → monetary_amount: "12 weeks within 15 weeks"
    - "242 days after start date" → temporal_text (not monetary): "242 days after start date"
    Use null if no quantifiable amount is present.

11. DEFINITIONS — How to handle glossary entries:
    If the chunk is purely a glossary entry (one or more terms defined), extract the MOST SIGNIFICANT definition. Set:
    - provision_type: "Definition"
    - deontic_type: null
    - subject.text: the term being defined (verbatim)
    - subject.suggested_cco_type: best guess (Resource for monetary terms, Role for actor terms, Unknown otherwise)
    - action_or_state: the definitional clause
    - extraction_rationale: note that this is a Definition and which term is being defined

    If a chunk has MULTIPLE definitions, extract the first substantive definition and note in extraction_rationale that other definitions exist in the chunk.

12. WHAT REMAINS NON_EXTRACTABLE — Explicit examples:
    Despite the broader scope, some chunks have no extractable regulatory content. Mark non_extractable for ALL of these patterns:

    (a) Empty cross-references:
        "See Chapter 8, Section 1.4."
        "Please refer to paragraph 222."

    (b) Pure transitional sentences:
        "This chapter outlines the tables used in the assessment process."
        "The following sections describe..."

    (c) Truncated fragments with no recoverable subject AND no recoverable rule:
        "funding band maximum for the standard that the apprentice started on, then"
        Note: This is an exception only if the chunk has NEITHER a clear subject NOR a clear rule. If the fragment contains a complete sentence with subject+verb, extract that.

    (d) Mid-cell table fragments without coherent statement:
        "| | 60 penalty units | | |"
        "| Non-punitive Withdrawal | form. An Appendix 7 is not required. |" — broken table cell
        Note: Truly fragmentary table cells with no readable rule should be non_extractable. A well-formed table row WITH a complete rule remains extractable.

    (e) Page-break artifacts and column overflow

    (f) Sentence fragments ending with ":" that introduce a list whose items are NOT in this chunk:
        "We will consider:"
        "Once the application outcome is received..."

    (g) POLICY ANNOUNCEMENTS describing past events without normative force:
        "In February 2025, the Government announced reforms..."
        "The Department has issued new guidance..."
        These describe historical events, not impose duties.

    (h) BULLET FRAGMENTS missing parent context:
        "• a declaration by a qualified accountant..." with no preceding sentence
        "• Evidenced, for the circumstances described in paragraph 74..." — past participle bullet
        If the bullet starts with a noun phrase or past participle and lacks a complete sentence (subject + modal/verb), mark non_extractable.

    (i) STAND-ALONE CONTEXTUAL STATEMENTS without normative force:
        "Another factor is whether the student has knowledge of the course requirements..."
        "Financial planning is recommended..."
        Pure background commentary; if the same chunk also contains a clear obligation, extract that — but don't extract the commentary itself.

    (j) PAST-STATE DESCRIPTIONS without future duty:
        "The provider has confirmed that the apprentice meets one of the conditions..."
        "The student received funding for two semesters and completed their first term..."
        These describe what already happened, not what must happen.

    When marking non_extractable, give a concrete reason citing which sub-category above applies (e.g., "policy announcement without normative force", "bullet fragment missing parent context", "past-state description without future duty").

13. PARSING TIPS:
    - A chunk may contain MULTIPLE provisions. Extract the MOST SIGNIFICANT one (the one that best characterizes the chunk). Note other provisions briefly in extraction_rationale.
    - A bullet marker at the start of a chunk does NOT disqualify it. Look for substantive content elsewhere in the chunk. BUT if the bullet AND surrounding context lack a complete subject+verb sentence, apply Rule 12(h).
    - Conditional clauses ("If X, then Y") indicate Conditional Norm Pattern — capture the condition AND the underlying provision_type.
    - Passive voice is acceptable for FundingAllocation, CalculationRule, EligibilityRule, and Definition. It is NOT acceptable for Obligation/Permission/Prohibition (those require named actors with modal verbs).

14. MODAL VERB GATE for deontic_type assignment:
    deontic_type=Obligation requires EITHER:
    (i) Explicit modal verb ("must", "shall", "is required to", "has to") with a named subject, OR
    (ii) provision_type is FundingAllocation/EligibilityRule/CalculationRule/Definition AND the chunk imposes a clear binding constraint (e.g., "Program must be at least 12 weeks" sets a binding eligibility rule).

    deontic_type=Permission requires EITHER:
    (i) Explicit permissive modal ("may", "can", "is permitted to", "is entitled to"), OR
    (ii) provision_type is FundingAllocation/EligibilityRule and the chunk grants an entitlement (e.g., "Students with dependents: maximum $740 per week" entitles recipients).

    deontic_type=Prohibition requires EITHER:
    (i) Explicit prohibitive modal ("must not", "shall not", "may not", "is prohibited from"), OR
    (ii) Explicit exclusion language ("not eligible", "excluded from", "is forbidden").

    deontic_type=null is REQUIRED when:
    - The chunk is purely a Definition (no deontic force)
    - The verb is advisory only ("are expected to", "should", "is recommended") — these are NOT Obligation
    - The verb is past tense describing a completed state ("has confirmed", "was approved", "received") — these are NOT Obligation
    - The verb is purely descriptive of a process ("is calculated", "is assessed", "is determined") — these are NOT Obligation unless the chunk also contains a clear binding rule

    Do NOT force deontic_type if no modal verb or binding constraint is genuinely present. null is acceptable.
"""

print('Stage 1 prompt defined (v3.1 — quality patches: anti-paraphrasing, fragment filter, modal verb gate).')

In [ ]:
# -- Stage 1 pipeline ---------------------------------------------------------
def resolve_pronoun_subject(decomposition: dict, jurisdiction: str) -> dict:
    """Post-process Stage 1 decomposition to resolve pronoun subjects.
    
    Behaviour:
      - Non-pronoun subjects: record resolution_method = 'none'
      - Pronoun subjects with jurisdiction mapping: resolve via DOCUMENT_ADDRESSEE
      - Pronoun subjects without mapping: mark as non_extractable
        (prevents 'you' from leaking into final KG as rdfs:label)
    
    Always preserves the LLM's original pronoun in subject.original_text.
    Idempotent: re-running on already-resolved output is a no-op.
    """
    if not isinstance(decomposition, dict):
        return decomposition
    if decomposition.get('extraction_decision') != 'extractable':
        return decomposition
    subj = decomposition.get('subject')
    if not isinstance(subj, dict):
        return decomposition
    # Idempotency: if already processed, skip
    if 'resolution_method' in subj:
        return decomposition
    
    raw_text = (subj.get('text') or '').strip()
    if not raw_text:
        return decomposition
    
    # Tokenise and check if subject is purely pronoun(s)
    tokens = re.findall(r"\b\w+\b", raw_text.lower())
    is_pronoun_only = bool(tokens) and all(t in PRONOUN_SET for t in tokens)
    
    if not is_pronoun_only:
        # Non-pronoun subject — record method 'none' for downstream consistency
        subj['original_text'] = raw_text
        subj['resolution_method'] = 'none'
        return decomposition
    
    # Pronoun case — attempt document-default resolution
    subj['original_text'] = raw_text
    mapped = DOCUMENT_ADDRESSEE.get(jurisdiction)
    if mapped:
        subj['text'] = mapped
        subj['resolution_method'] = 'document_default'
    else:
        # No mapping for this jurisdiction — filter as non_extractable
        # to prevent pronoun "you" leaking into final KG.
        subj['resolution_method'] = 'unresolved'
        decomposition['extraction_decision'] = 'non_extractable'
        decomposition['non_extractable_reason'] = f'Unresolved pronoun subject: "{raw_text}"'
    
    return decomposition


def run_stage1(retrieval_ctx: dict) -> dict:
    """Run Stage 1 (Deontic Decomposition) only.
    
    Returns a result dict that preserves full retrieval context so Stage 2
    can load this output and continue without the original retrieval JSON.
    """
    unit_id = retrieval_ctx['unit_id']
    jurisdiction = retrieval_ctx['jurisdiction']
    text = retrieval_ctx['text']
    section_path = ' > '.join(retrieval_ctx.get('section_path') or [])
    schema_str = format_schema_hits(retrieval_ctx.get('schema_hits', []))
    odp_str = format_odp_hits(retrieval_ctx.get('odp_hits', []))
    top_odp_label = (retrieval_ctx.get('odp_hits') or [{}])[0].get('label')
    
    result = {
        'unit_id': unit_id,
        'jurisdiction': jurisdiction,
        'unit_type': retrieval_ctx.get('unit_type', ''),
        'section_path': retrieval_ctx.get('section_path', []),
        'text': text,
        'schema_hits': retrieval_ctx.get('schema_hits', []),
        'odp_hits': retrieval_ctx.get('odp_hits', []),
        'top_odp_label': top_odp_label,
        'raw_stage1_response': None,
        'stage1_decomposition': None,
        'stage1_status': None,
        'stage1_error_message': None,
        'stage1_completed_at': None,
    }
    
    try:
        # Use retry wrapper for resilience to transient API failures
        raw_stage1 = call_stage1_llm_with_retry(
            STAGE1_SYSTEM,
            STAGE1_USER_TEMPLATE.format(
                unit_id=unit_id,
                jurisdiction=jurisdiction,
                unit_type=retrieval_ctx.get('unit_type', ''),
                section_path=section_path,
                text=text,
                schema_hits=schema_str,
                odp_hits=odp_str,
            ),
        )
        result['raw_stage1_response'] = raw_stage1
        
        if not (raw_stage1 or '').strip():
            result['stage1_status'] = 'error'
            result['stage1_error_message'] = 'Empty model response'
            return result
        
        decomposition = parse_json_response(raw_stage1)
        # Rule 11 post-processor: deterministic pronoun resolution
        decomposition = resolve_pronoun_subject(decomposition, jurisdiction)
        result['stage1_decomposition'] = decomposition
        
        decision = decomposition.get('extraction_decision')
        if decision == 'extractable':
            result['stage1_status'] = 'extractable'
        elif decision == 'non_extractable':
            result['stage1_status'] = 'non_extractable'
            result['stage1_error_message'] = decomposition.get('non_extractable_reason')
        else:
            result['stage1_status'] = 'error'
            result['stage1_error_message'] = f'Unexpected extraction_decision: {decision}'
        
    except Exception as e:
        result['stage1_status'] = 'error'
        result['stage1_error_message'] = f'Stage 1 failure: {e}'
    
    result['stage1_completed_at'] = datetime.now().isoformat()
    return result

print('Stage 1 pipeline defined.')

In [ ]:
# -- Load full corpus (no sampling) + shuffle for balanced distribution -------
final_contexts = load_final_retrieval_contexts()
sample_contexts = final_contexts

# Shuffle for balanced jurisdiction distribution
# (prevents single-jurisdiction dominance if run is interrupted)
random.Random(RANDOM_SEED).shuffle(sample_contexts)

print(f'Full corpus run: {len(sample_contexts)} chunks (shuffled with seed {RANDOM_SEED})')
print(f'\nPer-jurisdiction distribution:')
counts = Counter(ctx['jurisdiction'] for ctx in sample_contexts)
for jurisdiction in ['UK', 'Canada', 'Australia', 'USA']:
    print(f'  {jurisdiction:<10}: {counts.get(jurisdiction, 0)}')

# Verify shuffle worked - show first 10 chunk jurisdictions
print(f'\nFirst 10 chunks (shuffle verification):')
for i, ctx in enumerate(sample_contexts[:10], 1):
    print(f'  {i}. {ctx["unit_id"]} | {ctx["jurisdiction"]}')

# Estimate cost
est_cost = len(sample_contexts) * 0.0008  # V4-Pro slightly higher than V3.1
print(f'\nEstimated cost: ~${est_cost:.2f}')
print(f'Estimated runtime: ~{len(sample_contexts) * 1.0 / 60:.0f}-{len(sample_contexts) * 2.5 / 60:.0f} minutes')

In [ ]:
# -- Main loop with checkpointing + resume + ETA tracking ---------------------

out_path = OUTPUT_DIR / 'stage1_decompositions.json'

# Resume logic: load existing results if interrupted run
if out_path.exists():
    with open(out_path) as f:
        existing = json.load(f)
    stage1_results = existing.get('results', [])
    processed_ids = {r['unit_id'] for r in stage1_results}
    sample_contexts = [c for c in sample_contexts if c['unit_id'] not in processed_ids]
    status_counts = Counter(r['stage1_status'] for r in stage1_results)
    print(f'Resuming from checkpoint: {len(stage1_results)} done, {len(sample_contexts)} remaining')
    print(f'Existing status: {dict(status_counts)}')
else:
    stage1_results = []
    status_counts = Counter()
    print(f'Fresh run: {len(sample_contexts)} chunks to process')


def build_payload(results, counts):
    return {
        'metadata': {
            'stage': 'stage1_deontic_decomposition',
            'generation_model': GENERATION_MODEL,
            'api_provider': 'Together',
            'retrieval_source': str(RETRIEVAL_DIR),
            'total_chunks': len(results),
            'random_seed': RANDOM_SEED,
            'last_updated': datetime.now().isoformat(),
        },
        'status_counts': dict(counts),
        'results': results,
    }


# Main loop
start_time = time.time()
print(f'\nStarting at {datetime.now().strftime("%H:%M:%S")}...\n')

for i, ctx in enumerate(sample_contexts, start=1):
    result = run_stage1(ctx)
    stage1_results.append(result)
    status_counts[result['stage1_status']] += 1
    
    # Per-chunk print (only if VERBOSE or error or every Nth chunk)
    is_error = result['stage1_status'] == 'error'
    is_milestone = (i % PROGRESS_EVERY == 0)
    
    if VERBOSE or is_error or is_milestone:
        status = result['stage1_status']
        if status == 'extractable':
            decomp = result['stage1_decomposition'] or {}
            print(f'[{i}/{len(sample_contexts)}] {ctx["unit_id"]} | {ctx["jurisdiction"]} '
                  f'| extractable | {decomp.get("deontic_type")} | {decomp.get("likely_primary_odp_label")}')
        elif status == 'non_extractable':
            print(f'[{i}/{len(sample_contexts)}] {ctx["unit_id"]} | {ctx["jurisdiction"]} '
                  f'| non_extractable: {result["stage1_error_message"][:80]}')
        else:
            print(f'[{i}/{len(sample_contexts)}] {ctx["unit_id"]} | {ctx["jurisdiction"]} '
                  f'| ERROR: {result["stage1_error_message"]}')
    
    # Checkpoint save + ETA
    if i % CHECKPOINT_INTERVAL == 0:
        save_json(out_path, build_payload(stage1_results, status_counts))
        elapsed = time.time() - start_time
        rate = i / elapsed if elapsed > 0 else 0
        remaining = (len(sample_contexts) - i) / rate if rate > 0 else 0
        print(f'  --- CHECKPOINT @ {i}: ext={status_counts.get("extractable",0)} '
              f'non_ext={status_counts.get("non_extractable",0)} '
              f'err={status_counts.get("error",0)} '
              f'| rate={rate:.1f}/s | ETA={remaining/60:.1f}min ---')

# Final save
save_json(out_path, build_payload(stage1_results, status_counts))

elapsed = time.time() - start_time
print(f'\nDONE in {elapsed/60:.1f} min')
print(f'Saved: {out_path}')
print(f'\nFinal status counts:')
for status, count in status_counts.most_common():
    print(f'  {status:<20}: {count}')
print(f'\nExtraction rate: {status_counts.get("extractable", 0)}/{len(stage1_results)} '
      f'= {100*status_counts.get("extractable", 0)/max(len(stage1_results),1):.1f}%')

In [ ]:
# -- Deep inspection of one Stage 1 result -----------------------------------
INSPECT_INDEX = 0

if stage1_results:
    r = stage1_results[INSPECT_INDEX]
    print('=' * 70)
    print(f'UNIT ID     : {r["unit_id"]}')
    print(f'JURISDICTION: {r["jurisdiction"]}')
    print(f'UNIT TYPE   : {r["unit_type"]}')
    print(f'STATUS      : {r["stage1_status"]}')
    print(f'ERROR       : {r.get("stage1_error_message")}')
    print(f'TOP ODP     : {r.get("top_odp_label")}')
    print('=' * 70)
    
    print('\n--- PROVISION TEXT ---')
    print(r['text'])
    
    print('\n--- TOP 3 SCHEMA HITS (input to Stage 1) ---')
    for h in r.get('schema_hits', [])[:3]:
        print(f'  [{h["label"]}] {h["score"]:.4f} ({h["item_type"]})')
    
    print('\n--- TOP 3 ODP HITS (input to Stage 1) ---')
    for h in r.get('odp_hits', [])[:3]:
        print(f'  [{h["label"]}] {h["score"]:.4f}')
    
    print('\n--- STAGE 1 RAW RESPONSE ---')
    print(r.get('raw_stage1_response'))
    
    print('\n--- STAGE 1 DECOMPOSITION (parsed) ---')
    print(json.dumps(r.get('stage1_decomposition'), indent=2, ensure_ascii=False))
else:
    print('No results to inspect.')

In [ ]:
# -- Quality scan across all results ------------------------------------------
print('=' * 70)
print(f'STAGE 1 QUALITY SCAN — {len(stage1_results)} chunks')
print('=' * 70)

issues_found = []
for r in stage1_results:
    decomp = r.get('stage1_decomposition') or {}
    unit_id = r['unit_id']
    flags = []
    
    # Check 1: status sensible
    if r['stage1_status'] == 'error':
        flags.append(f'ERROR: {r.get("stage1_error_message")}')
    
    # Check 2: "null" as string instead of JSON null
    string_null_fields = []
    for k, v in decomp.items():
        if isinstance(v, str) and v.strip().lower() == 'null':
            string_null_fields.append(k)
    if string_null_fields:
        flags.append(f'string "null" in fields: {string_null_fields}')
    
    # Check 3: ODP label has "ODP-x |" prefix
    primary = decomp.get('likely_primary_odp_label')
    if isinstance(primary, str) and '|' in primary:
        flags.append(f'primary ODP has prefix: {primary!r}')
    secondaries = decomp.get('likely_secondary_odp_labels') or []
    bad_secondaries = [s for s in secondaries if isinstance(s, str) and '|' in s]
    if bad_secondaries:
        flags.append(f'secondary ODPs have prefix: {bad_secondaries}')
    
    # Check 4: ODP label not in known set
    if isinstance(primary, str) and primary not in ODP_LABELS and primary is not None:
        stripped = primary.split('|')[-1].strip() if '|' in primary else primary
        if stripped not in ODP_LABELS:
            flags.append(f'unknown primary ODP: {primary!r}')
    
    # Check 5: deontic type validity
    deontic = decomp.get('deontic_type')
    if deontic is not None and deontic not in DEONTIC_TYPES:
        flags.append(f'invalid deontic_type: {deontic!r}')
    
    # Check 6: subject's suggested_cco_type validity
    subject = decomp.get('subject') or {}
    sub_type = subject.get('suggested_cco_type') if isinstance(subject, dict) else None
    if sub_type and sub_type not in SUBJECT_CCO_TYPES:
        flags.append(f'invalid subject CCO type: {sub_type!r}')
    
    if flags:
        issues_found.append((unit_id, flags))
        # Only print first 20 issues to avoid Jupyter overflow
        if len(issues_found) <= 20:
            print(f'\n{unit_id}:')
            for f in flags:
                print(f'  - {f}')

if not issues_found:
    print('\nNo format/compliance issues detected.')
elif len(issues_found) > 20:
    print(f'\n... and {len(issues_found) - 20} more issues (truncated for display)')

print(f'\n--- Summary ---')
print(f'Total results       : {len(stage1_results)}')
print(f'Results with issues : {len(issues_found)}')
print(f'Results clean       : {len(stage1_results) - len(issues_found)}')

In [ ]:
# -- Pronoun resolution audit -------------------------------------------------
print('=' * 70)
print('PRONOUN RESOLUTION AUDIT')
print('=' * 70)

method_by_jur = defaultdict(Counter)
unresolved_units = []
resolved_units = []

for r in stage1_results:
    # Check both extractable AND non_extractable (unresolved now becomes non_ext)
    decomp = r.get('stage1_decomposition') or {}
    subj = decomp.get('subject') or {}
    if not isinstance(subj, dict):
        continue
    method = subj.get('resolution_method')
    if not method:
        continue
    method_by_jur[r['jurisdiction']][method] += 1
    if method == 'document_default':
        resolved_units.append((r['unit_id'], subj.get('original_text'), subj.get('text')))
    elif method == 'unresolved':
        unresolved_units.append((r['unit_id'], r['jurisdiction'], subj.get('original_text')))

print(f"\nBy jurisdiction:")
for jur in sorted(method_by_jur.keys()):
    print(f"  {jur}:")
    for method, count in method_by_jur[jur].most_common():
        print(f"    {method:20s}: {count}")

print(f"\nResolved (document_default): {len(resolved_units)}")
for unit_id, orig, resolved in resolved_units[:20]:  # show first 20
    print(f"  {unit_id}: '{orig}' -> '{resolved}'")
if len(resolved_units) > 20:
    print(f"  ... and {len(resolved_units) - 20} more")

if unresolved_units:
    print(f"\nUnresolved (now filtered as non_extractable): {len(unresolved_units)}")
    for unit_id, jur, orig in unresolved_units[:20]:
        print(f"  {unit_id} ({jur}): '{orig}'")
    if len(unresolved_units) > 20:
        print(f"  ... and {len(unresolved_units) - 20} more")
else:
    print(f"\nNo unresolved pronouns.")

In [ ]:
# -- Stage 1 Validation Stats ---------------------------------------------
# Verify Stage 1 output health before moving to Stage 2

import json
from collections import Counter
from pathlib import Path

OUTPUT = Path("/Users/umair/CCO-GRO/3 - Extraction and  Validation Layer/output/stage1_decomposition/stage1_decompositions.json")
data = json.loads(OUTPUT.read_text())
results = data["results"]

print("=" * 70)
print(f"STAGE 1 VALIDATION REPORT")
print(f"Total chunks: {len(results)}")
print("=" * 70)


# === 1. JURISDICTION BALANCE ===
print("\n[1] JURISDICTION BALANCE")
print("-" * 70)
juris_total = Counter(r['jurisdiction'] for r in results)
juris_ext = Counter(r['jurisdiction'] for r in results if r['stage1_status'] == 'extractable')
juris_nonext = Counter(r['jurisdiction'] for r in results if r['stage1_status'] == 'non_extractable')

print(f"{'Jurisdiction':<15} {'Total':>7} {'Extract':>8} {'Non-Ext':>8} {'Rate':>7}")
for j in sorted(juris_total):
    rate = juris_ext[j] / juris_total[j] * 100
    print(f"{j:<15} {juris_total[j]:>7} {juris_ext[j]:>8} {juris_nonext[j]:>8} {rate:>6.1f}%")
total_rate = sum(juris_ext.values()) / sum(juris_total.values()) * 100
print(f"{'TOTAL':<15} {sum(juris_total.values()):>7} {sum(juris_ext.values()):>8} {sum(juris_nonext.values()):>8} {total_rate:>6.1f}%")


# === 2. PROVISION TYPE DISTRIBUTION ===
print("\n[2] PROVISION TYPE DISTRIBUTION (extractable only)")
print("-" * 70)
ptypes = Counter(r['stage1_decomposition'].get('provision_type') 
                 for r in results if r['stage1_status'] == 'extractable')
total_ext = sum(ptypes.values())
print(f"{'Provision Type':<25} {'Count':>7} {'%':>7}")
for pt, c in ptypes.most_common():
    pct = c / total_ext * 100
    print(f"{str(pt):<25} {c:>7} {pct:>6.1f}%")


# === 3. DEONTIC TYPE DISTRIBUTION ===
print("\n[3] DEONTIC TYPE DISTRIBUTION (extractable only)")
print("-" * 70)
dtypes = Counter(r['stage1_decomposition'].get('deontic_type') 
                 for r in results if r['stage1_status'] == 'extractable')
print(f"{'Deontic Type':<25} {'Count':>7} {'%':>7}")
for dt, c in dtypes.most_common():
    pct = c / total_ext * 100
    print(f"{str(dt):<25} {c:>7} {pct:>6.1f}%")


# === 4. PRIMARY ODP DISTRIBUTION ===
print("\n[4] PRIMARY ODP DISTRIBUTION (extractable only)")
print("-" * 70)
odps = Counter(r['stage1_decomposition'].get('likely_primary_odp_label') 
               for r in results if r['stage1_status'] == 'extractable')
print(f"{'ODP Label':<45} {'Count':>7} {'%':>7}")
for odp, c in odps.most_common():
    pct = c / total_ext * 100
    print(f"{str(odp):<45} {c:>7} {pct:>6.1f}%")


# === 5. SECONDARY ODP USAGE ===
print("\n[5] SECONDARY ODP USAGE (any position)")
print("-" * 70)
sec_odps = Counter()
for r in results:
    if r['stage1_status'] == 'extractable':
        for odp in r['stage1_decomposition'].get('likely_secondary_odp_labels', []) or []:
            sec_odps[odp] += 1
print(f"{'ODP Label':<45} {'Count':>7}")
for odp, c in sec_odps.most_common():
    print(f"{str(odp):<45} {c:>7}")


# === 6. CCO SUBJECT TYPE DISTRIBUTION ===
print("\n[6] CCO SUBJECT TYPE DISTRIBUTION (extractable only)")
print("-" * 70)
cco_types = Counter()
for r in results:
    if r['stage1_status'] == 'extractable':
        subj = r['stage1_decomposition'].get('subject', {})
        if subj:
            cco_types[subj.get('suggested_cco_type')] += 1
print(f"{'CCO Type':<25} {'Count':>7} {'%':>7}")
for ct, c in cco_types.most_common():
    pct = c / total_ext * 100
    print(f"{str(ct):<25} {c:>7} {pct:>6.1f}%")


# === 7. MONETARY AMOUNT CAPTURE ===
print("\n[7] MONETARY AMOUNTS")
print("-" * 70)
monetary_count = sum(1 for r in results 
                     if r['stage1_status'] == 'extractable' 
                     and r['stage1_decomposition'].get('monetary_amount'))
print(f"Chunks with monetary_amount captured: {monetary_count} / {total_ext} ({monetary_count/total_ext*100:.1f}%)")

print("\nSample monetary amounts:")
samples = [r for r in results 
           if r['stage1_status'] == 'extractable' 
           and r['stage1_decomposition'].get('monetary_amount')]
for r in samples[:10]:
    amt = r['stage1_decomposition']['monetary_amount']
    uid = r['unit_id']
    print(f"  {uid}: {amt[:70]}")


# === 8. NON-EXTRACTABLE REASONS ===
print("\n[8] NON-EXTRACTABLE REASONS (top 15)")
print("-" * 70)
reasons = Counter()
for r in results:
    if r['stage1_status'] == 'non_extractable':
        reason = r['stage1_error_message'] or "unknown"
        # Normalize to first 60 chars
        reasons[reason[:60].strip()] += 1
total_nonext = sum(reasons.values())
print(f"{'Reason':<60} {'Count':>6} {'%':>7}")
for reason, c in reasons.most_common(15):
    pct = c / total_nonext * 100
    print(f"{reason:<60} {c:>6} {pct:>6.1f}%")


# === 9. PRONOUN RESOLUTION STATS ===
print("\n[9] PRONOUN RESOLUTION (extractable only)")
print("-" * 70)
res_methods = Counter()
res_unresolved = []
for r in results:
    if r['stage1_status'] == 'extractable':
        subj = r['stage1_decomposition'].get('subject', {})
        method = subj.get('resolution_method', 'none')
        res_methods[method] += 1
print(f"{'Resolution Method':<25} {'Count':>7}")
for m, c in res_methods.most_common():
    print(f"{str(m):<25} {c:>7}")


# === 10. PRONOUN-RELATED NON-EXTRACTABLES ===
print("\n[10] PRONOUN-RELATED NON-EXTRACTABLES")
print("-" * 70)
pronoun_filtered = [r for r in results 
                    if r['stage1_status'] == 'non_extractable' 
                    and r['stage1_error_message'] 
                    and 'pronoun' in r['stage1_error_message'].lower()]
print(f"Non-extractable due to unresolved pronoun: {len(pronoun_filtered)}")
print(f"\nBreakdown by jurisdiction:")
juris_pronoun = Counter(r['jurisdiction'] for r in pronoun_filtered)
for j, c in juris_pronoun.most_common():
    print(f"  {j}: {c}")
print(f"\nSample pronouns that failed resolution:")
for r in pronoun_filtered[:8]:
    msg = r['stage1_error_message']
    print(f"  {r['unit_id']} ({r['jurisdiction']}): {msg[:80]}")


# === 11. EMPTY/INVALID EXTRACTIONS (sanity check) ===
print("\n[11] SANITY CHECKS")
print("-" * 70)
no_subject = 0
no_norm_statement = 0
norm_fallback = 0
for r in results:
    if r['stage1_status'] == 'extractable':
        decomp = r['stage1_decomposition']
        subj = decomp.get('subject', {})
        if not subj or not subj.get('text'):
            no_subject += 1
        if not decomp.get('norm_statement'):
            no_norm_statement += 1
        if decomp.get('deontic_type') == 'Norm':
            norm_fallback += 1

print(f"Extractable chunks with no subject:       {no_subject} ({no_subject/total_ext*100:.1f}%)")
print(f"Extractable chunks with no norm_statement: {no_norm_statement} ({no_norm_statement/total_ext*100:.1f}%)")
print(f"Deontic_type=Norm fallback usage:          {norm_fallback} ({norm_fallback/total_ext*100:.1f}%)")


# === 12. UNIT TYPE BREAKDOWN ===
print("\n[12] UNIT TYPE EXTRACTION RATE")
print("-" * 70)
unit_total = Counter(r['unit_type'] for r in results)
unit_ext = Counter(r['unit_type'] for r in results if r['stage1_status'] == 'extractable')
print(f"{'Unit Type':<25} {'Total':>7} {'Extract':>8} {'Rate':>7}")
for ut in sorted(unit_total):
    rate = unit_ext[ut] / unit_total[ut] * 100 if unit_total[ut] else 0
    print(f"{ut:<25} {unit_total[ut]:>7} {unit_ext[ut]:>8} {rate:>6.1f}%")


print("\n" + "=" * 70)
print("VALIDATION REPORT COMPLETE")
print("=" * 70)